# 생성모델을 이용한 신약설계 — ADMET 예측과 MPO 최종 스코어링 (5일차 실습)

4일차에서 생성한 분자를 이어받아, **ADMET(흡수·분포·대사·배설·독성)을 예측**하고, 웹툴(SwissADME)과 **교차 비교**한 뒤, 여러 목표를 하나로 합치는 **MPO 최종 스코어링**으로 후보를 선별한다.

이 흐름은 신약개발의 **DMTA 사이클**(Design–Make–Test–Analyze)의 dry(in silico) 버전이다:
- **Design** = 4일차 생성모델이 만든 분자
- **Make** = 합성가능성(SA score)으로 "만들 수 있는가"를 가늠
- **Test** = ADMET-AI 예측 (wet-lab 실험을 대신하는 dry Test)
- **Analyze** = MPO 스코어링 + 독성 게이트로 후보 선별 → 다음 사이클 설계에 반영

> **실행 순서:** 위에서부터 셀을 순서대로 실행한다. GPU(T4) 런타임을 권장한다([런타임]→[런타임 유형 변경]→T4 GPU).


## 0. 설치

ADMET 예측에는 **ADMET-AI**(pip 설치)를, 마지막 MPO 스코어링에는 **REINVENT4**(4일차와 동일)를 사용한다.

### 0-1. ADMET-AI 설치

ADMET-AI는 TDC(Therapeutics Data Commons) ADMET 벤치마크로 학습된 Chemprop 기반 예측 도구다. SMILES만 넣으면 약 41개 ADMET 항목과 "승인약 대비 백분위"를 한 번에 예측한다. pip 한 줄로 설치되고 CPU로도 동작한다.

In [ ]:
# ADMET-AI 설치 — torch·chemprop을 함께 받으므로 몇 분 걸립니다.
!pip -q install admet-ai rdkit pandas matplotlib mols2grid

# ※ 4일차와 달리 이 실습에는 REINVENT4가 필요하지 않습니다.
#    MPO 점수는 5절에서 RDKit으로 직접 계산합니다.
#    (admet-ai가 torch를 최신 버전으로 올려 버려서 REINVENT가 쓰는
#     torchvision과 충돌합니다 — 같은 세션에서 둘을 함께 쓰지 않습니다.)

### 0-2. 설치 확인

다음 셀에서 `admet-ai 준비 완료` 가 찍히면 정상입니다.
설치 직후 Colab이 세션 재시작을 요구하면, 재시작한 뒤 이 셀부터 이어서 실행하세요.

In [ ]:
import sys
import numpy as np, pandas as pd, rdkit, torch
from admet_ai import ADMETModel

print("python  ", sys.version.split()[0])
print("numpy   ", np.__version__)
print("pandas  ", pd.__version__)
print("rdkit   ", rdkit.__version__)
print("torch   ", torch.__version__)
print("-" * 40)
print("admet-ai 준비 완료")

## 1. 입력 — 4일차 생성분자 불러오기

4일차 실습에서 생성한 분자 CSV(`trxr_generated.csv`)를 이어받는다. 찾는 순서는 다음과 같다.

1. **같은 세션에서 4일차를 이어서 하는 경우** — `/content/REINVENT4/trxr_generated.csv`
2. **4일차를 건너뛴 경우** — 강의 저장소에서 같은 파일을 내려받는다 (모두 동일한 분자로 실습)
3. 둘 다 실패하면 예시 분자로 대체한다

In [ ]:
import os, subprocess
import pandas as pd
from rdkit import Chem

ASSETS_URL = ("https://github.com/eunjoolee122/2026-aidrugdiscovery"
              "/releases/download/assets-v1/assets.zip")


def canon(s):
    """SMILES를 정규형(canonical)으로 통일. 파싱 실패하면 None."""
    m = Chem.MolFromSmiles(str(s))
    return Chem.MolToSmiles(m) if m else None


# ── 1) 같은 세션에서 4일차를 이어서 하는 경우 ──
gen_csv = "/content/REINVENT4/trxr_generated.csv"

# ── 2) 없으면 강의 저장소에서 4일차 결과를 받아온다 ──
if not os.path.exists(gen_csv):
    gen_csv = "/content/trxr_generated.csv"
    if not os.path.exists(gen_csv):
        print("4일차 생성분자를 강의 저장소에서 내려받는 중 ...")
        subprocess.run(["wget", "-q", "-O", "/content/assets.zip", ASSETS_URL])
        subprocess.run(["unzip", "-o", "-q", "/content/assets.zip",
                        "trxr_generated.csv", "-d", "/content"])

# ── 3) 그래도 없으면 예시 분자 ──
if os.path.exists(gen_csv):
    raw = pd.read_csv(gen_csv)["SMILES"].dropna().tolist()
    print(f"4일차 생성분자 {len(raw)}개 로드  ({gen_csv})")
else:
    raw = ["CC(=O)Oc1ccccc1C(=O)O", "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
           "CC(C)Cc1ccc(cc1)C(C)C(=O)O", "CCN(CC)CCOC(=O)c1ccc(N)cc1",
           "COc1ccc2nc(N)sc2c1", "O=C(O)c1ccccc1", "CN(C)CCc1c[nH]c2ccccc12",
           "OC(=O)Cc1ccc(O)cc1", "Cc1ccccc1NC(=O)c1ccccc1", "c1ccc(-c2ccccc2)cc1"]
    print(f"4일차 결과를 찾지 못해 예시 분자 {len(raw)}개 사용")
    print("  ※ 예시는 기존 승인약이라 '생성분자의 ADMET'이라는 취지와는 다릅니다.")

# 유효 분자만 정규형으로 통일 (CPU 예측 시간을 감안해 200개로 제한)
smiles_list = [c for s in raw if (c := canon(s))][:200]
print(f"유효 분자 {len(smiles_list)}개로 진행")

## 2. ADMET-AI로 ADMET 예측 (Test)

SMILES 리스트를 넣으면 흡수·분포·대사·배설·독성 전 범위를 예측한다. 분류 항목(hERG·AMES·DILI·CYP·BBB 등)은 0~1 확률, 물성·회귀 항목(용해도·반감기·LD50 등)은 실제 예측값, 그리고 각 항목의 "승인약 대비 백분위(_drugbank_approved_percentile)"가 함께 나온다.

In [ ]:
from admet_ai import ADMETModel

model = ADMETModel()
admet = model.predict(smiles=smiles_list).reset_index().rename(columns={"index": "SMILES"})
print("예측 완료:", admet.shape, "(분자 x 항목)")
# 주요 항목만 미리보기
show = ["SMILES","QED","logP","tpsa","Solubility_AqSolDB","BBB_Martins","hERG","AMES","DILI"]
admet[show].head()

### 주요 항목 읽는 법

| 구분 | 항목(예) | 의미 |
|---|---|---|
| 흡수 | Caco2_Wang, HIA_Hou, PAMPA_NCATS, Pgp_Broccatelli, Bioavailability_Ma | 장 투과·흡수·수송체 |
| 분포 | BBB_Martins, PPBR_AZ, VDss_Lombardo | 뇌 투과·혈장 단백 결합·분포용적 |
| 대사 | CYP1A2/2C19/2C9/2D6/3A4_Veith(저해), *_Substrate | CYP 효소 저해·기질 |
| 배설 | Clearance_*, Half_Life_Obach | 청소율·반감기 |
| 독성 | hERG, AMES, DILI, ClinTox, Carcinogens, LD50_Zhu, NR-*/SR-*(Tox21) | 심장·유전·간독성 등 |
| 물성 | molecular_weight, logP, tpsa, QED, Lipinski | 약물성 지표 |

> 분류 항목은 값이 **높을수록 해당 성질(독성 포함) 확률이 크다**. 예: hERG 0.8 → 심장독성 위험 높음.

## 3. 예측 결과 시각화

생성 분자 집합의 핵심 ADMET 항목 분포와 독성 위험 요약을 그린다.

In [ ]:
import matplotlib.pyplot as plt

key = ["QED", "logP", "Solubility_AqSolDB", "BBB_Martins", "hERG", "DILI"]
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, c in zip(axes.ravel(), key):
    ax.hist(admet[c], bins=30)
    ax.set_title(c); ax.set_ylabel("count")
plt.suptitle("생성 분자의 주요 ADMET 항목 분포")
plt.tight_layout(); plt.show()

In [ ]:
# 독성 위험(확률 > 0.5) 분자 수 요약
tox_cols = ["hERG", "AMES", "DILI", "ClinTox", "Carcinogens_Lagunin"]
high_risk = (admet[tox_cols] > 0.5).sum().sort_values(ascending=False)
print("독성 위험(확률 > 0.5) 분자 수:")
print(high_risk.to_string())

ax = high_risk.plot(kind="bar", figsize=(7,4), title="독성 항목별 고위험 분자 수 (prob > 0.5)")
ax.set_ylabel("분자 수"); plt.tight_layout(); plt.show()

In [ ]:
# 대표 분자 1개의 "승인약 대비 백분위" — 기존 약과 어디쯤인지
pct_cols = ["QED_drugbank_approved_percentile", "logP_drugbank_approved_percentile",
            "tpsa_drugbank_approved_percentile", "hERG_drugbank_approved_percentile",
            "DILI_drugbank_approved_percentile", "AMES_drugbank_approved_percentile"]
one = admet.iloc[0]
vals = [one[c] for c in pct_cols]
labels = [c.replace("_drugbank_approved_percentile", "") for c in pct_cols]
plt.figure(figsize=(8,4))
plt.barh(labels, vals); plt.xlim(0, 100); plt.xlabel("승인약 대비 백분위 (%)")
plt.title(f"대표 분자의 ADMET 백분위\n{one['SMILES']}")
plt.tight_layout(); plt.show()

## 4. SwissADME 웹툴과 교차 비교 (상위 몇 개)

같은 분자를 **다른 방법론의 웹툴(SwissADME)** 로도 예측해 결과가 얼마나 일치하는지 확인한다. SwissADME는 API가 없어 자동화가 안 되므로, **상위 몇 개 분자만 수동으로** 돌려 비교한다.

**진행 방법**
1. 아래 셀을 실행해 상위 5개 분자의 SMILES를 출력한다.
2. [swissadme.ch](http://www.swissadme.ch/) 에 접속 → 왼쪽 입력창에 SMILES를 붙여넣고 **Run** 클릭.
3. 결과 표 아래 **CSV** 버튼으로 결과를 내려받는다.
4. Colab 왼쪽 [파일] 탭에 그 CSV를 업로드한다(파일명 예: `swissadme.csv`).

In [ ]:
# 상위 5개(QED 기준) 분자 SMILES — SwissADME에 붙여넣기
topN = admet.sort_values("QED", ascending=False).head(5).reset_index(drop=True)
print("아래 5줄을 SwissADME 입력창에 붙여넣으세요:\n")
for s in topN["SMILES"]:
    print(s)

In [ ]:
# SwissADME 결과 CSV를 업로드했을 때만 비교합니다.
# ※ 열 이름은 SwissADME 버전에 따라 다르므로 sw.columns로 확인 후 맞춥니다.
import os

if not os.path.exists("swissadme.csv"):
    print("swissadme.csv 가 없습니다 — 이 셀은 건너뛰고 다음으로 진행하세요.")
    print("(왼쪽 [파일] 탭에 업로드한 뒤 다시 실행하면 비교표가 나옵니다.)")
else:
    sw = pd.read_csv("swissadme.csv")
    print("SwissADME 열:", list(sw.columns)[:15], "...")

    def pick(df, *names):
        for n in names:
            if n in df.columns:
                return df[n].values
        return [None] * len(df)

    n = min(len(topN), len(sw))   # 입력 순서가 같다고 가정
    compare = pd.DataFrame({
        "SMILES":        topN["SMILES"].values[:n],
        "logP_ADMET-AI": topN["logP"].values[:n].round(2),
        "logP_Swiss":    pick(sw, "Consensus Log P", "iLOGP", "XLOGP3")[:n],
        "TPSA_ADMET-AI": topN["tpsa"].values[:n].round(1),
        "TPSA_Swiss":    pick(sw, "TPSA")[:n],
    })
    display(compare)

> **해석 포인트:** 두 도구는 학습 데이터·방법이 다르므로 값이 정확히 같지는 않다. 경향(순위)이 일치하는지, 특정 분자에서 크게 어긋나는지를 본다. 도구마다 예측이 다를 수 있다는 점이 곧 **예측 신뢰도(적용범위)** 를 조심해야 하는 이유다.

## 5. MPO 최종 스코어링 — 직접 계산 (Analyze 1/2)

여러 목표(약물성 QED · 분자량 · 지용성 logP · 구조경보)를 **하나의 점수로 합치는** 것이
다중 목표 최적화(MPO)다. 4일차에서 REINVENT가 해 주던 계산을, 여기서는 **RDKit으로 직접**
해 본다. 공식이 눈에 보이면 "점수가 왜 이렇게 나왔는지"를 따질 수 있다.

**세 단계로 이루어진다.**

| 단계 | 하는 일 | 예 |
|---|---|---|
| ① 원시값 계산 | 분자에서 물성을 뽑는다 | MW = 474.6, logP = 1.61 |
| ② transform | 원시값을 **0~1 점수**로 바꾼다 | MW 474.6 → 0.912 |
| ③ 가중 결합 | 점수들을 하나로 합친다 | geometric_mean → 0.726 |

②의 `double_sigmoid` 는 **[low, high] 범위 안일수록 1에 가까운** 종 모양 함수다.
분자량은 200~500, logP는 1~5를 목표 범위로 둔다. 범위를 벗어날수록 점수가 급격히 떨어진다.

③의 `geometric_mean`(기하평균)은 **하나라도 0점이면 전체가 0점**이 되는 성질이 있어,
"모든 조건을 동시에 만족하라"는 압력을 만든다. 산술평균이라면 한 항목이 0이어도
다른 항목이 높으면 살아남는데, 그건 신약 후보 선별에 맞지 않는다.

구조경보(alerts)는 가중평균에 들어가지 않고 **최종 점수에 곱해지는 필터**다 —
걸리면 다른 점수가 아무리 좋아도 0점이 된다.

> 아래 구현은 REINVENT4의 scoring 결과와 **소수점 7자리까지 일치**하도록 맞춘 것이다.

In [ ]:
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, QED

# ── 구조경보 (걸리면 0점) ──────────────────────────────────────────
ALERT_SMARTS = ["[*;r{8-17}]",   # 8~17원 거대고리
                "[#8][#8]",      # 과산화물
                "[#6;+]",        # 탄소 양이온
                "[#16][#16]",    # 이황화물
                "C#C"]           # 알카인
_ALERTS = [Chem.MolFromSmarts(s) for s in ALERT_SMARTS]


# ── transform ─────────────────────────────────────────────────────
def _sigmoid(x, k):
    h = np.clip(k * np.asarray(x, float) * np.log(10), -700, 700)
    return np.where(h >= 0, 1 / (1 + np.exp(-h)), np.exp(h) / (1 + np.exp(h)))


def double_sigmoid(x, low, high, coef_div, coef_si, coef_se):
    """[low, high] 범위 안이면 1에 가깝고, 벗어나면 0으로 떨어진다."""
    x = np.asarray(x, float)
    center = (high - low) / 2 + low
    out = np.zeros_like(x)
    left = x < center
    out[left] = _sigmoid(x[left] - low, coef_si / coef_div)
    out[~left] = 1 - _sigmoid(x[~left] - high, coef_se / coef_div)
    return out


def geometric_mean(scores, weights):
    """가중 기하평균 — 하나라도 0이면 전체가 0에 가까워진다."""
    s = np.maximum(np.array(scores, float), 1e-8)
    w = np.array(weights, float)
    w = w / w.sum()
    return np.prod(s ** w[:, None], axis=0)


# ── ① 원시값 ──────────────────────────────────────────────────────
mols = [Chem.MolFromSmiles(s) for s in smiles_list]
raw_qed = np.array([QED.qed(m) for m in mols])
raw_mw = np.array([Descriptors.MolWt(m) for m in mols])
raw_logp = np.array([Crippen.MolLogP(m) for m in mols])

# ── ② transform ───────────────────────────────────────────────────
s_qed = raw_qed                                              # 이미 0~1
s_mw = double_sigmoid(raw_mw, 200, 500, 500, 20, 20)         # 목표 200~500
s_logp = double_sigmoid(raw_logp, 1, 5, 5, 20, 20)           # 목표 1~5
s_alert = np.array([0.0 if any(m.HasSubstructMatch(p) for p in _ALERTS)
                    else 1.0 for m in mols])

# ── ③ 가중 결합 + 구조경보 필터 ───────────────────────────────────
mpo_score = geometric_mean([s_qed, s_mw, s_logp], [0.5, 0.25, 0.25]) * s_alert

mpo = pd.DataFrame({
    "SMILES": smiles_list, "MPO": mpo_score,
    "QED": raw_qed, "MW": raw_mw, "SlogP": raw_logp,
    "s_MW": s_mw, "s_SlogP": s_logp, "Alerts": s_alert,
})

print(f"MPO 점수 계산 완료 — {len(mpo)}개")
print(f"구조경보에 걸린 분자: {int((s_alert == 0).sum())}개")
mpo.sort_values("MPO", ascending=False).head().round(3)

## 6. Analyze — 독성 게이트 + 최종 랭킹 (Analyze 2/2)

ADMET-AI의 독성 예측으로 위험 분자를 먼저 걸러내고(**독성 게이트**), 통과한 분자를 REINVENT MPO 점수로 정렬해 **최종 후보**를 뽑는다. 합성가능성(SA score)도 참고 지표로 함께 본다.

In [ ]:
from rdkit.Chem import RDConfig
import sys, os
sys.path.append(os.path.join(RDConfig.RDContribDir, "SA_Score"))
import sascorer

# ADMET 예측과 MPO 점수를 canonical SMILES 기준으로 합친다
admet["key"] = admet["SMILES"].map(canon)
mpo["key"] = mpo["SMILES"].map(canon)
merged = admet.merge(mpo[["key", "MPO"]], on="key", how="inner")
print(f"병합된 분자: {len(merged)}개")

# 합성가능성 (SA score — 낮을수록 만들기 쉬움, 1~10)
merged["SA"] = [sascorer.calculateScore(Chem.MolFromSmiles(s))
                for s in merged["SMILES"]]

# ── 독성 게이트 ───────────────────────────────────────────────────
TOX_CUT = 0.5      # 통과 분자가 너무 적으면 0.6~0.7로 올려 본다
gate = ((merged["hERG"] < TOX_CUT) &
        (merged["AMES"] < TOX_CUT) &
        (merged["DILI"] < TOX_CUT))
print(f"독성 게이트(< {TOX_CUT}) 통과: {int(gate.sum())} / {len(merged)}")

if gate.sum() == 0:
    print("\n  통과 분자가 없습니다.")
    print("  · TOX_CUT을 0.6~0.7로 올리거나")
    print("  · 항목을 hERG 하나로 줄여 다시 실행해 보세요.")
    print("  일단 게이트 없이 MPO 순위만 보겠습니다.")
    final = merged.sort_values("MPO", ascending=False).reset_index(drop=True)
else:
    final = merged[gate].sort_values("MPO", ascending=False).reset_index(drop=True)

n_alert = int((final["MPO"] == 0).sum())
if n_alert:
    print(f"  ※ 구조경보에 걸려 MPO=0인 분자가 {n_alert}개 있습니다 (목록 맨 아래).")

cols = ["SMILES", "MPO", "QED", "SA", "hERG", "AMES", "DILI",
        "logP", "Solubility_AqSolDB"]
final[[c for c in cols if c in final.columns]].head(10).round(3)

In [ ]:
# 최종 상위 후보 구조 보기
import mols2grid
top = final.head(12).copy()
top["mol"] = [Chem.MolFromSmiles(s) for s in top["SMILES"]]
mols2grid.display(top, mol_col="mol",
                  subset=["MPO", "QED", "hERG"],
                  transform={"MPO": lambda x: f"{x:.2f}", "QED": lambda x: f"{x:.2f}",
                             "hERG": lambda x: f"{x:.2f}"},
                  n_items_per_page=12, size=(200, 200))

## 7. 정리 — DMTA 사이클로 보기

이번 실습은 생성모델 결과를 **한 번의 dry DMTA 사이클**로 흘려보낸 것이다.

| 단계 | 이번 실습에서 | 도구 |
|---|---|---|
| **Design** | 4일차 생성모델이 후보 분자 생성 | REINVENT4 (4일차) |
| **Make** | 합성가능성(SA score)으로 만들기 난이도 가늠 | RDKit |
| **Test** | ADMET 예측으로 흡수·독성 등 평가 | ADMET-AI (+ SwissADME 교차검증) |
| **Analyze** | 독성 게이트 + MPO 점수로 후보 선별 | ADMET-AI + RDKit MPO |

**다음 사이클 제안:** Analyze에서 탈락한 이유(예: hERG 독성, 낮은 용해도)를 Design에
되먹인다 — 4일차 생성 RL의 reward에 해당 물성 목표를 추가하거나, 통과 분자로
transfer learning을 수행해 더 나은 후보를 만든다.

> 이번 실습은 DMTA의 **dry(in silico)** 절반이다. 실제 합성·생물학적 시험(wet)과
> 이를 자동화하는 **self-driving lab / AI-로봇 통합**은 강의(이론)에서 다룬다.

### 🎯 이제 여러분 차례
- MPO의 **weight**(0.5 / 0.25 / 0.25)나 **목표 범위**(MW 200~500, logP 1~5)를 바꿔
  후보 순위가 어떻게 달라지는지 관찰
- `geometric_mean` 을 산술평균으로 바꿔 보고, 어떤 분자가 새로 올라오는지 확인
- 독성 게이트 기준(0.5)을 조정하거나 다른 항목(ClinTox·Carcinogens) 추가
- ADMET-AI가 준 백분위로 "기존 승인약과 비슷한 프로파일"의 분자를 골라보기